# Phase 9 : présentation et soutenance

Projet final Machine Learning, blocs 6 et 8.

## Objectif de ce notebook

Deux choses, et la première n'est pas de la rédaction.

**Vérifier la checklist finale des livrables par le code.** Le guide en donne six catégories. Les
cocher à la main la veille de la soutenance est le meilleur moyen de découvrir un fichier
manquant devant le jury. La section 1 les contrôle une par une et échoue bruyamment si quelque
chose manque.

**Produire `docs/soutenance.md`**, qui contient l'executive summary, le contenu des douze
diapositives, les trois chiffres clés, les recommandations priorisées, les limites avec leur
mitigation, l'auto-évaluation et la préparation aux questions du jury.

## Le message principal, en une phrase

> Tous les avantages précoces ne se valent pas, et le classement auquel tout le monde se fie est
> presque exactement à l'envers.

Tout le reste de la soutenance en découle : les trois chiffres la démontrent, les recommandations
en sortent, et le modèle sert à la quantifier.

Données fournies par Oracle's Elixir (Tim Sevenhuysen, oracleselixir.com).

## 1. Vérification de la checklist finale

In [1]:
import json
import sys
from datetime import date
from pathlib import Path

import joblib
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config, viz

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
fr, nb = viz.entier_fr, viz.nombre_fr

donnees = pd.read_parquet(config.DATA_PROCESSED / "lol_at15.parquet")
pipeline = joblib.load(config.MODELS / "pipeline_final.joblib")

figures = sorted(config.FIGURES.glob("*.png"))
carnets = sorted((config.ROOT / "notebooks").glob("*.ipynb"))
exports = sorted((config.ROOT / "data" / "exports").glob("lolwin_at15_*"))

FEATURES = (config.NUMERIC_FEATURES + config.CATEGORICAL_FEATURES
            + config.BOOLEAN_FEATURES)


def echantillon_pour_prediction(n=5):
    """Cinq lignes brutes, typees comme la pipeline les attend."""
    X = donnees[FEATURES].head(n).copy()
    numeriques = config.NUMERIC_FEATURES + config.BOOLEAN_FEATURES
    X[numeriques] = X[numeriques].astype("float64")
    X[config.CATEGORICAL_FEATURES] = X[config.CATEGORICAL_FEATURES].astype("str")
    return X


def carnet_execute(chemin):
    """Un notebook livrable doit etre execute, sans cellule vide ni trace d'erreur."""
    contenu = json.loads(chemin.read_text(encoding="utf-8"))
    cellules = [c for c in contenu["cells"] if c["cell_type"] == "code"]
    sans_execution = [c for c in cellules if c.get("execution_count") is None]
    erreurs = [o for c in cellules for o in c.get("outputs", [])
               if o.get("output_type") == "error"]
    return not sans_execution and not erreurs


controles = [
    ("Dataset final au format parquet",
     (config.DATA_PROCESSED / "lol_at15.parquet").exists()),
    ("Exports alternatifs, csv et xlsx", len(exports) >= 3),
    ("Data dictionary present", (config.DOCS / "data_dictionary.md").exists()),
    ("Guide de reproductibilite", (config.DOCS / "reproductibilite.md").exists()),
    ("Changelog", (config.ROOT / "CHANGELOG.md").exists()),
    ("Neuf notebooks livres", len(carnets) == 9),
    # On exclut le notebook 09 du controle : il est justement en train de tourner, donc
    # son fichier sur disque n'a pas encore de sorties. Lui se verifie autrement, en
    # arrivant au bout de sa derniere cellule sans exception.
    ("Les huit autres notebooks executes sans erreur",
     all(carnet_execute(c) for c in carnets if not c.name.startswith("09"))),
    ("Au moins 7 visualisations", len(figures) >= 7),
    ("Pipeline complete sauvegardee, preprocessing inclus",
     [nom for nom, _ in pipeline.steps] == ["preprocessing", "modele"]),
    ("Le modele recharge et predit",
     len(pipeline.predict_proba(echantillon_pour_prediction())) == 5),
    ("Rapport de modelisation", (config.DOCS / "rapport_modelisation.md").exists()),
    ("Rapport d'analyse", (config.DOCS / "rapport_analytique.md").exists()),
    ("Journal d'utilisation de l'IA", (config.DOCS / "journal_ia.md").exists()),
    ("Cadrage initial", (config.DOCS / "00_cadrage.md").exists()),
]

bilan = pd.DataFrame(controles, columns=["livrable", "ok"])
bilan["statut"] = bilan["ok"].map({True: "ok", False: "MANQUANT"})
print(bilan[["livrable", "statut"]].to_string(index=False))
print()
print(f"{len(figures)} figures, {len(carnets)} notebooks, {len(exports)} exports")

# Un message qui dit quoi faire plutot qu'un assert sec : sur un depot tout juste clone,
# les artefacts derives n'existent pas encore. La bonne reaction, c'est d'executer les
# notebooks qui les produisent, pas de chercher un bug ici.
if not bilan["ok"].all():
    absents = bilan.loc[~bilan["ok"], "livrable"].tolist()
    raise AssertionError(
        f"Livrables manquants : {absents}. Sur un depot fraichement clone, executer les "
        "notebooks 01 a 08 dans l'ordre avant celui-ci : ils produisent le dataset, le "
        "modele et les exports."
    )

                                           livrable statut
                    Dataset final au format parquet     ok
                   Exports alternatifs, csv et xlsx     ok
                            Data dictionary present     ok
                          Guide de reproductibilite     ok
                                          Changelog     ok
                              Neuf notebooks livres     ok
     Les huit autres notebooks executes sans erreur     ok
                          Au moins 7 visualisations     ok
Pipeline complete sauvegardee, preprocessing inclus     ok
                       Le modele recharge et predit     ok
                            Rapport de modelisation     ok
                                  Rapport d'analyse     ok
                      Journal d'utilisation de l'IA     ok
                                    Cadrage initial     ok

19 figures, 9 notebooks, 3 exports


Le contrôle qui compte est l'avant-dernier : la pipeline rechargée doit prédire. Un `.joblib` qui
existe mais ne se recharge pas est un livrable manquant qui a l'air présent.

## 2. Les chiffres de la soutenance

Ils sont rassemblés en un seul endroit plutôt que recopiés d'une diapositive à l'autre. Un chiffre
saisi deux fois est un chiffre qui finira par diverger, et se tromper d'une décimale devant un
jury coûte plus cher que la décimale.

In [2]:
CHIFFRES = {
    "lignes": len(donnees),
    "parties": donnees["gameid"].nunique(),
    "periode": f"{donnees['date'].min():%Y} a {donnees['date'].max():%Y}",
    "ligues": donnees["league"].nunique(),
    "features": len(FEATURES),
    # Phase 5, mesures a avantage economique egal
    "dragon_or": 1030, "dragon_points": 14.1,
    "sang_points": 0.4, "sang_p": 0.77,
    "heraut_brut": 22.9, "heraut_points": 0.9,
    "pente_or": 13.6, "bleu_taux": 52.9, "bleu_or": 215,
    "compo_ecart_max": 2.8,
    "forme_serree": 18.1, "forme_avance": 12.4,
    "tier1_conversion": 80.1, "tier3_conversion": 80.5, "tier_p": 0.56,
    # Phase 6, regimes de partie
    "part_serree": 29, "part_large": 28, "taux_serree": 57, "taux_large": 92,
    # Phase 7, modele
    "baseline_or_test": 74.5, "modele_accuracy": 75.9, "modele_auc": 0.844,
    "modele_logloss": 0.487, "gain_points": 1.4, "gain_lignes": 229,
    "lignes_test": 16546, "ecart_familles": 0.003,
    "rang_dragon": 2, "rang_sang": 7,
}

print(f"Volume   : {fr(CHIFFRES['lignes'])} lignes, {fr(CHIFFRES['parties'])} parties, "
      f"{CHIFFRES['periode']}, {CHIFFRES['ligues']} ligues")
print(f"Modele   : {nb(CHIFFRES['modele_accuracy'])} % d'accuracy, "
      f"{nb(CHIFFRES['modele_auc'], 3)} d'AUC")
print(f"Baseline : {nb(CHIFFRES['baseline_or_test'])} % pour la regle economique")
print(f"Gain     : {nb(CHIFFRES['gain_points'])} point, "
      f"{fr(CHIFFRES['gain_lignes'])} lignes sur {fr(CHIFFRES['lignes_test'])}")

Volume   : 92 616 lignes, 46 308 parties, 2022 a 2026, 82 ligues
Modele   : 75,9 % d'accuracy, 0,844 d'AUC
Baseline : 74,5 % pour la regle economique
Gain     : 1,4 point, 229 lignes sur 16 546


### Le contrôle qui manquait

Les chiffres ci-dessus sont saisis à la main, alors que `docs/rapport_modelisation.md` est
généré par le notebook 07. C'est le seul endroit du projet où la règle « les documents sont
produits par le code qui produit les données » n'est pas tenue, et donc le seul endroit où un
ré-entraînement pourrait rendre trois documents faux en silence : le dossier de soutenance, le
support de présentation et le guide d'explication.

La cellule suivante ferme cette brèche. Elle relit le rapport généré, en extrait les mesures du
modèle final, et échoue si elles ne correspondent pas aux valeurs saisies ici.

In [3]:
# On verifie que les chiffres tapes a la main collent au rapport genere par le notebook
# 07.
rapport = (config.DOCS / "rapport_modelisation.md").read_text(encoding="utf-8")

ligne_modele = next(l for l in rapport.splitlines() if l.startswith("| Modele final"))
ligne_baseline = next(l for l in rapport.splitlines()
                      if l.startswith("| Baseline plus riche"))

mesures = [c.strip() for c in ligne_modele.split("|") if c.strip()]
accuracy_rapport = float(mesures[1])
auc_rapport = float(mesures[2])
baseline_rapport = float([c.strip() for c in ligne_baseline.split("|") if c.strip()][1])

coherence = [
    ("accuracy du modele", round(100 * accuracy_rapport, 1), CHIFFRES["modele_accuracy"]),
    ("AUC du modele", round(auc_rapport, 3), CHIFFRES["modele_auc"]),
    ("baseline economique", round(100 * baseline_rapport, 1), CHIFFRES["baseline_or_test"]),
    ("gain en points", round(100 * (accuracy_rapport - baseline_rapport), 1),
     CHIFFRES["gain_points"]),
]

table = pd.DataFrame(coherence, columns=["mesure", "rapport_genere", "saisi_a_la_main"])
table["accord"] = table["rapport_genere"] == table["saisi_a_la_main"]
print(table.to_string(index=False))

desaccords = table.loc[~table["accord"], "mesure"].tolist()
if desaccords:
    raise AssertionError(
        f"Les chiffres de la soutenance ne correspondent plus au rapport genere : "
        f"{desaccords}. Mettre a jour CHIFFRES, docs/soutenance_deck.html et "
        "docs/guide_explication.md avant de presenter."
    )
print()
print("Les chiffres du dossier de soutenance correspondent au rapport de modelisation.")

             mesure  rapport_genere  saisi_a_la_main  accord
 accuracy du modele          75.900           75.900    True
      AUC du modele           0.844            0.844    True
baseline economique          74.500           74.500    True
     gain en points           1.400            1.400    True

Les chiffres du dossier de soutenance correspondent au rapport de modelisation.


## 3. Les trois chiffres à retenir

Le guide demande des chiffres surprenants, significatifs, simples et mémorables. Ces trois-là
forment une progression : le premier surprend, le deuxième renverse, le troisième désarme.

| # | Chiffre | Message | Impact |
|---|---|---|---|
| 1 | **1 030 or** | Ce que vaut le premier dragon, à avantage économique égal | Une priorité d'entraînement chiffrée, comparable au coût d'une rotation |
| 2 | **0** | Ce que vaut le premier sang, dans les mêmes conditions | L'indicateur le plus commenté du jeu n'apporte rien au-delà de l'or qu'il rapporte |
| 3 | **+1,4 point** | Ce que le modèle ajoute à une règle d'une ligne | L'honnêteté du gain, et le déplacement du débat vers la calibration |

Le troisième est délibérément à mon désavantage. C'est celui qui rend les deux premiers
crédibles : un candidat qui annonce lui-même la faiblesse de son gain n'a pas maquillé le reste.

## 4. Génération du dossier de soutenance

In [4]:
c = CHIFFRES
inventaire_figures = "\n".join(f"- `figures/{f.name}`" for f in figures)

dossier = f"""# Dossier de soutenance

Document genere par `notebooks/09_soutenance.ipynb`, a ne pas editer a la main.

Projet : prediction de l'issue d'une partie professionnelle de League of Legends a la 15e minute.
Donnees fournies par Oracle's Elixir (Tim Sevenhuysen, oracleselixir.com).

---

## Executive summary

**Contexte.** Dans une partie professionnelle de League of Legends, les quinze premieres minutes
pesent lourd sans tout decider. Les staffs et les plateformes de statistiques veulent savoir
quels avantages precoces se convertissent reellement en victoire, et lesquels sont surestimes.

**Objectif.** Determiner ce que vaut chaque avantage disponible a la 15e minute, et construire un
modele qui affiche une probabilite de victoire credible a cet instant.

**Methodologie.** {fr(c['lignes'])} lignes equipe, {fr(c['parties'])} parties professionnelles de
{c['periode']}, {c['ligues']} ligues, croisees avec les tags de champions de Riot Data Dragon et
un referentiel de ligues construit a la main. {c['features']} variables retenues, toutes connues
a la 15e minute. Entrainement sur 2022-2025, test sur 2026 ouvert une seule fois.

**Principaux resultats.**

1. **Le premier dragon vaut environ {fr(c['dragon_or'])} or.** A ecart d'or nul, l'equipe qui
   l'obtient gagne {nb(c['dragon_points'])} points de plus. Il ne rapporte pourtant aucun or.
2. **Le premier sang ne vaut rien au-dela de l'or qu'il rapporte** : {nb(c['sang_points'])} point
   a avantage egal, p = {nb(c['sang_p'], 2)}. Le premier heraut non plus, alors qu'il mene le
   classement brut avec {nb(c['heraut_brut'])} points d'ecart.
3. **Le modele atteint {nb(c['modele_accuracy'])} % d'accuracy et {nb(c['modele_auc'], 3)} d'AUC**
   contre {nb(c['baseline_or_test'])} % pour la regle « le plus riche a 15 gagne », soit
   {nb(c['gain_points'])} point de gain. Son apport reel est la probabilite calibree, que la
   baseline ne sait pas produire.

**Recommandations.** Prioriser la contestation du premier dragon sur la recherche du premier
sang. Afficher le dragon plutot que le premier sang sur un habillage de diffusion. Concentrer le
travail de macro sur les parties serrees, qui representent {c['part_serree']} % des situations et
ou l'ecart de niveau pese le plus.

**Limites.** Association n'est pas causalite : le controle porte sur l'or, pas sur le niveau des
equipes. Le perimetre exclut une grande partie de la LPL sur la periode d'entrainement.

---

## Structure de la presentation

### Diapositive 1, titre

Predire l'issue d'une partie professionnelle de League of Legends a la 15e minute.
Projet final Machine Learning, blocs 6 et 8.

### Diapositive 2, contexte et objectif

- Les quinze premieres minutes pesent lourd, sans tout decider
- Question : quels avantages precoces se convertissent vraiment en victoire ?
- Public vise : analystes et coachs, plateformes de statistiques, casteurs
- Enjeu : un staff qui sait ce que vaut un dragon n'entraine pas comme un staff qui l'ignore

### Diapositive 3, les donnees

- Trois sources, trois formats : Oracle's Elixir (CSV), Riot Data Dragon (JSON), referentiel de
  ligues (XLSX)
- {fr(c['lignes'])} lignes equipe, {fr(c['parties'])} parties, {c['periode']}, {c['ligues']} ligues
- Defi qualite principal : un export de fin de partie, ou la majorite des colonnes contiennent le
  resultat. Instant de prediction fige a la 15e minute, tout le reste supprime
- Regle d'inclusion fondee sur la completude mesuree, jamais sur un nom de ligue

### Diapositive 4, insight 1, la hierarchie des avantages

- Question : quels avantages pesent le plus a la 15e minute ?
- Figure : `06_hierarchie_avantages.png`
- Chiffre : l'ecart d'or atteint 0,82 d'AUC a lui seul, les objectifs neutres entre 0,58 et 0,61
- Implication : le classement brut semble clair, et il est trompeur

### Diapositive 5, insight 2, le renversement

- Question : que reste-t-il de chaque objectif, a avantage economique egal ?
- Figure : `06_illusion_des_objectifs.png`, la diapositive centrale de la soutenance
- Chiffres : premier heraut {nb(c['heraut_brut'])} points en brut, {nb(c['heraut_points'])} a or
  egal. Premier dragon {nb(c['dragon_points'])} points a or egal, soit {fr(c['dragon_or'])} or
- Implication : le classement est presque exactement inverse. Mecanique : le sang et le heraut
  rapportent de l'or, deja compte ; le dragon n'en rapporte aucun

### Diapositive 6, insight 3, ce qui ne compte pas

- Questions : la composition de draft, le niveau de ligue
- Figures : `06_composition_sans_effet.png`, `06_ligues_conversion.png`
- Chiffres : ecart maximal de {nb(c['compo_ecart_max'])} points pour la composition, tous les
  intervalles contiennent 50 %. Tier 1 {nb(c['tier1_conversion'])} % contre tier 3
  {nb(c['tier3_conversion'])} %, p = {nb(c['tier_p'], 2)}
- Implication : deux idees recues desamorcees. Ne pas arbitrer un pick sur un comptage de tags,
  ne pas decoter un joueur de ligue mineure sur sa conversion

### Diapositive 7, insight 4, ou se joue la partie

- Question : la forme recente aide-t-elle a convertir une avance ?
- Figures : `06_trois_regimes_de_partie.png`, `06_forme_par_regime.png`
- Chiffres : {c['part_serree']} % des parties encore serrees a 15 minutes, le camp en avance n'y
  gagne que {c['taux_serree']} %. L'amplitude de la forme vaut {nb(c['forme_serree'])} points en
  partie serree contre {nb(c['forme_avance'])} points quand l'equipe mene
- Implication : la forme mesure le niveau, pas une capacite a conclure. L'ecart de niveau se joue
  dans les parties serrees

### Diapositive 8, insight 5, le cote bleu

- Question : l'avantage du cote bleu existe-t-il encore ?
- Figure : `06_avantage_cote_bleu.png`
- Chiffre : {nb(c['bleu_taux'])} % de victoires, soit environ {fr(c['bleu_or'])} or offerts avant
  le premier pick, stable sur quatre saisons
- Implication : parametre fixe a integrer a la strategie de draft, pas a reviser chaque patch

### Diapositive 9, le modele predictif

- Cible : `result`, classification binaire, classes equilibrees a 50 %
- Metrique principale : ROC AUC. Secondaires : accuracy pour la communication, log loss pour la
  calibration, car le livrable est une probabilite affichee et non un verdict
- Trois baselines calculees avant tout modele, la plus forte a {nb(c['baseline_or_test'])} % sur 2026
- Trois familles comparees par `GridSearchCV` avec `TimeSeriesSplit` : regression logistique,
  foret aleatoire, gradient boosting. Elles tiennent en {nb(c['ecart_familles'], 3)} d'AUC
- **La regression logistique gagne** : la mieux calibree, et 38 lignes d'ecart
  entrainement/validation contre 584 pour la foret
- Verdict sur 2026, ouvert une seule fois : {nb(c['modele_accuracy'])} % d'accuracy,
  {nb(c['modele_auc'], 3)} d'AUC, {nb(c['modele_logloss'], 3)} de log loss
- Figures : `07_courbe_roc.png`, `07_calibration.png`, `07_coefficients_logistique.png`
- Interpretation : `firstdragon` ressort au rang {c['rang_dragon']} sur 34, confirmant la mesure
  de la phase 5 par une methode independante
- Limites : le modele est le moins performant sur le tier 1, la ou les remontees sont les plus
  frequentes

### Diapositive 10, synthese

| Chiffre | Message |
|---|---|
| **{fr(c['dragon_or'])} or** | Ce que vaut le premier dragon a avantage economique egal |
| **0** | Ce que vaut le premier sang dans les memes conditions |
| **{nb(c['gain_points'])} point** | Ce que le modele ajoute a une regle d'une ligne |

Message principal : tous les avantages precoces ne se valent pas, et le classement auquel tout le
monde se fie est presque exactement a l'envers.

### Diapositive 11, recommandations

Voir la section dediee ci-dessous.

### Diapositive 12, annexes

Methodologie detaillee, limites, `docs/reproductibilite.md`, inventaire des figures.

---

## Recommandations priorisees

### Recommandation 1, priorite haute

**Insight source.** A ecart d'or nul, le premier dragon apporte {nb(c['dragon_points'])} points de
taux de victoire, soit {fr(c['dragon_or'])} or, avec une p-valeur de l'ordre de 10 puissance
moins 26. Le premier sang en apporte {nb(c['sang_points'])}, non significatif.

**Action.** Reecrire la priorite d'early game : contester systematiquement le premier dragon,
cesser de rechercher le premier sang lorsqu'il coute une position sur la carte.

**Resultat attendu.** Un premier dragon supplementaire vaut l'equivalent de {fr(c['dragon_or'])}
or d'avance, soit environ {nb(c['dragon_points'])} points de taux de victoire sur les parties
concernees.

**Prerequis.** Aucun. La consigne est immediatement applicable en scrim.

### Recommandation 2, priorite haute

**Insight source.** Le premier sang et le premier heraut dominent le classement brut et ne valent
rien une fois l'or neutralise.

**Action.** Pour une plateforme de statistiques ou un habillage de diffusion, remplacer
l'indicateur « premier sang » par « premier dragon » dans les elements affiches en direct.

**Resultat attendu.** Un indicateur affiche qui porte reellement de l'information, au lieu d'un
indicateur qui duplique l'ecart d'or deja affiche a cote.

**Prerequis.** Modification de l'habillage graphique.

### Recommandation 3, priorite moyenne

**Insight source.** {c['part_serree']} % des parties restent indecises a la 15e minute, et
l'amplitude de la forme y vaut {nb(c['forme_serree'])} points contre {nb(c['forme_avance'])}
points dans les parties deja engagees.

**Action.** Concentrer le travail de macro et de prise de decision sur les situations serrees
plutot que sur la conversion des avances.

**Resultat attendu.** Gain sur le segment ou l'ecart de niveau se joue reellement.

**Prerequis.** Selection de scrims et de revues video ciblees sur ce regime.

### Recommandation 4, priorite basse

**Insight source.** Tier 1 et tier 3 convertissent une avance de plus de 1 000 or de facon
statistiquement indiscernable, p = {nb(c['tier_p'], 2)}.

**Action.** En recrutement, ne pas decoter les statistiques de conversion d'un joueur venant
d'une ligue mineure.

**Resultat attendu.** Elargissement du vivier sans perte de qualite sur ce critere.

**Prerequis.** Aucun.

---

## Limites, avec mitigation

### Limite 1, association et non causalite

**Description.** Le controle a or egal neutralise l'avantage economique, pas le niveau des
equipes. Une equipe plus forte prend plus souvent le dragon.

**Impact potentiel.** La valeur de {fr(c['dragon_or'])} or attribuee au dragon est un majorant :
elle melange la valeur de l'objectif et la force de qui l'obtient.

**Mitigation appliquee.** La comparaison est faite dans une bande ou l'ecart d'or moyen entre
groupes reste sous 15 or, verifie et non suppose. La formulation retenue est « associe a » et non
« cause ».

**Pour lever la limite.** Un indicateur de niveau independant, type classement Elo, absent des
sources du projet.

### Limite 2, perimetre incomplet

**Description.** La regle d'inclusion exige un snapshot complet a 15 minutes sur les deux lignes
d'une partie. Elle a ecarte une grande partie de la LPL sur 2022-2025.

**Impact potentiel.** La Chine passe de 0,2 % du jeu d'entrainement a 6,4 % du jeu de test, et le
modele y obtient sa plus mauvaise AUC regionale. Aucune conclusion regionale sur la LPL n'est
possible.

**Mitigation appliquee.** Le filtrage porte sur un critere mesure et documente, jamais sur un nom
de ligue. La derive est mesuree et publiee dans le rapport de modelisation.

**Pour lever la limite.** Reentrainer des que la LPL est disponible en volume sur plusieurs
saisons.

### Limite 3, representation grossiere de la draft

**Description.** La composition est reduite a un comptage de tags Data Dragon. La synergie entre
champions et l'ordre des picks sont absents d'Oracle's Elixir.

**Impact potentiel.** Le resultat negatif de la question 3 porte sur cette representation, pas
sur la draft en general.

**Mitigation appliquee.** La conclusion est formulee comme telle dans le rapport d'analyse.

**Pour lever la limite.** Une source contenant l'ordre de draft.

### Limite 4, un modele qui ignore la structure des parties

**Description.** Chaque ligne equipe est predite independamment. Les deux probabilites d'une meme
partie ne somment donc a 1 qu'approximativement, avec 220 parties ou les deux equipes sont
donnees gagnantes.

**Impact potentiel.** Visible sur un affichage en direct.

**Mitigation appliquee.** L'incoherence est mesuree et publiee plutot que passee sous silence.

**Pour lever la limite.** Normaliser les deux probabilites, ou reformuler le probleme au niveau
de la partie avec des variables en difference.

### Limite 5, derive du meta

**Description.** Quatre saisons de meta sont agregees a l'entrainement, dont deux changements de
regles majeurs.

**Impact potentiel.** Un modele unique est moins juste sur une periode heterogene.

**Mitigation appliquee.** `patch_seq` remplace le numero de patch, `ecart_or_normalise` corrige la
derive economique, et les categorielles instables sont exclues.

**Pour lever la limite.** Reentrainement periodique, a chaque debut de saison.

---

## Auto-evaluation

| Critere | Note sur 5 | Commentaire |
|---|---|---|
| Cadrage | 5 | Question predictive, cible, metrique et risque de fuite definis avant toute ligne de code |
| Extraction | 4 | Trois sources, trois formats. Le referentiel de ligues repose a 16 % sur une identification de confiance moyenne, tracee par une colonne dediee |
| Diagnostic qualite | 5 | Cinq dimensions, et le defaut le plus grave trouve dans une colonne parfaitement remplie, `turretplates` |
| Nettoyage | 5 | Cinq fuites rattrapees par un controle empirique de correlation, `firsttower` ecartee contre le cadrage initial |
| Transformation | 5 | Onze variables construites, fuite evitee sur la mediane de patch par fenetre expansive |
| Analyse | 4 | Cinq questions traitees avec intervalles de confiance. Le controle porte sur l'or seul, pas sur le niveau |
| Visualisation | 5 | Dix-neuf figures, un tableau de bord, titres formules comme des messages |
| Modelisation | 5 | Trois baselines avant tout modele, trois familles, test ouvert une fois, modele simple retenu |
| Documentation | 5 | Documents generes par le code, empreinte publiee, audit des dependances |
| Utilisation de l'IA | 4 | Prompts et corrections journalises, y compris les affirmations reecrites apres mesure |

### Reflexion metacognitive

**Ce que j'ai appris.** Qu'une correlation ne repond presque jamais a la question posee. Le
premier sang correle a 0,199 avec la victoire et ne vaut rien : il a fallu comparer a or egal
pour le voir. C'est la lecon la plus transferable du projet.

**L'etape la plus difficile.** La phase 3. Une liste de colonnes interdites ecrite de memoire est
insuffisante sur un export de fin de partie ; c'est un test empirique, comparer la correlation de
chaque colonne a celle de `golddiffat15`, qui a rattrape cinq oublis.

**Ce dont je suis le plus fier.** Que la regression logistique gagne. Le modele le plus simple est
aussi le mieux calibre et le moins sujet au surapprentissage, et c'est celui dont on peut lire les
coefficients un par un devant un jury.

**Ce que je ferais differemment.** L'audit des dependances en phase 1 et non en phase 8. Il a
revele que `requirements.txt` decrivait un environnement qui n'etait pas celui de l'execution,
incoherence qui a vecu tout le projet sans consequence visible.

**Transfert professionnel.** La discipline du jeu de test ouvert une seule fois, et l'habitude de
verifier une affirmation plutot que de la supposer, valent bien au-dela de ce sujet.

---

## Preparation aux questions du jury

| Question probable | Reponse |
|---|---|
| Pourquoi la 15e minute et pas la 10e ou la 20e ? | Oracle's Elixir fournit des snapshots a 10, 15, 20 et 25 minutes. La 15e est le dernier instant ou une majorite de parties reste indecise, {c['part_serree']} % sous 1 000 or d'ecart, tout en portant deja du signal |
| Pourquoi avoir ecarte `firsttower` alors que le cadrage la gardait ? | Deux mesures. Elle est attribuee dans 100 % des parties, la ou les trois autres objectifs laissent des parties sans titulaire, et elle correle a 0,391 contre 0,18 a 0,25. En jeu professionnel la premiere tourelle tombe souvent apres la 15e minute |
| Pourquoi `TimeSeriesSplit` plutot qu'un `KFold` ? | Un `KFold` remelange l'ordre chronologique a l'interieur du train et reproduit a petite echelle le probleme que le split chronologique corrige : le modele validerait sur des parties anterieures a certaines de ses parties d'entrainement |
| Le gain de {nb(c['gain_points'])} point justifie-t-il un modele ? | En accuracy, difficilement. Mais la baseline ne produit pas de probabilite, seulement un verdict binaire. Le livrable metier est une probabilite affichee, et le modele atteint {nb(c['modele_logloss'], 3)} de log loss avec une calibration proche de la diagonale |
| Pourquoi le coefficient de `firstblood` est-il negatif ? | Il se lit conditionnellement, pas causalement. A ecart d'or, d'experience et de kills donnes, le residu du premier sang n'a plus de valeur. C'est coherent avec la phase 5, qui mesurait {nb(c['sang_points'])} point non significatif a or egal |
| Le split aleatoire donne un meilleur score, pourquoi garder le chronologique ? | Parce que les deux protocoles n'evaluent pas la meme population : le test chronologique ne contient que 2026, homogene, le test aleatoire tire dans cinq saisons heterogenes. Le choix se justifie par la condition d'usage, predire des parties futures, pas par le score |
| Comment savez-vous qu'il n'y a pas de fuite ? | Trois garde-fous. Une deny-list centralisee dans `src/config.py`, un test empirique de correlation qui a rattrape cinq oublis, et un audit verifiant qu'aucune variable historique n'est renseignee sur la premiere partie d'un groupe. Le resultat, {nb(c['modele_accuracy'])} % d'accuracy, est dans le domaine annonce avant modelisation |
| Que feriez-vous avec plus de temps ? | L'ordre de la draft et un classement Elo par equipe. Trois familles de modeles se tiennent en {nb(c['ecart_familles'], 3)} d'AUC : le plafond vient de l'information disponible, pas de l'algorithme |

---

## Inventaire des figures

{inventaire_figures}
"""

chemin = config.DOCS / "soutenance.md"
chemin.write_text(dossier, encoding="utf-8")
print(f"Dossier ecrit : {chemin}")
print(f"{fr(len(dossier))} caracteres")

Dossier ecrit : E:\LWP(LoLWinPrediciton)\lol-win-prediction\docs\soutenance.md
17 883 caracteres


## 5. Le fil de l'oral, sept minutes

Le guide donne un découpage. Celui-ci l'adapte au fait que le renversement des objectifs est une
meilleure accroche que n'importe quelle introduction méthodologique.

| Temps | Contenu | Ce qu'il faut dire |
|---|---|---|
| 0:00-0:30 | Accroche | « Le premier sang est l'indicateur le plus commenté de League of Legends. Il ne vaut rien. » Puis le contexte en deux phrases |
| 0:30-1:00 | Problématique | Quels avantages précoces se convertissent vraiment, et à qui cela sert |
| 1:00-3:00 | Trois insights | Le renversement des objectifs, les 29 % de parties serrées, les deux résultats négatifs |
| 3:00-5:00 | Le modèle | Baselines, trois familles, la logistique qui gagne, le verdict unique sur 2026, et les limites annoncées avant qu'on les demande |
| 5:00-6:00 | Recommandations | Les deux de priorité haute, chiffrées |
| 6:00-7:00 | Conclusion et questions | Les trois chiffres, puis le message principal |

**Le piège à éviter.** Commencer par la méthodologie. Le travail de nettoyage et d'anti-fuite est
la partie la plus solide du projet, et c'est précisément pour cela qu'il faut le garder pour les
questions : il répond mieux qu'il n'introduit.

**La phrase de secours**, si le jury demande ce qu'il faut retenir : le classement des avantages
précoces auquel tout le monde se fie est presque exactement à l'envers, et il suffit de comparer
à or égal pour le voir.